# THERMA SEARCH AI · 03 · WiSARD veri seti keşfi

## Aşama 1 — Kaynak, lisans ve deney planı (arşiv gerekmez)
[Resmî WiSARD sayfası](https://sites.google.com/uw.edu/wisard/) · *WiSARD: A Labeled Visual and Thermal Image Dataset for Wilderness Search and Rescue*, IROS 2022 · [DOI 10.1109/IROS47612.2022.9981298](https://doi.org/10.1109/IROS47612.2022.9981298).

Kullanıcının doğruladığı **resmî sayfa beyanı**, yerel ölçüm değildir:

| Alt küme | Kaynak beyanı |
|---|---|
| Visual Only | 26.862 etiketli görüntü |
| Thermal Only | 29.989 etiketli görüntü |
| Multi-modal | 15.453 eşzamanlı RGB–termal çift; diğer iki alt kümenin içinden |
| Tam arşiv | Yaklaşık 40,54 GB — bu notebook'ta kullanılmaz/indirilmez |
| Multi-modal sample | Yaklaşık 971,6 MB; **tek veri toplama uçuşu** |

Çiftler diğer alt kümelerde yer aldığı için bu sayılar birbirine eklenmez. Örnek arşivin dosya kimliği/sürümü, iç yapısı, görüntü boyutları ve etiket şeması bu teslimde gerçek veriyle doğrulanmadı.

**Lisans:** Bu geliştirme ortamında resmî sayfa isteği 403 verdi. Veri lisansı metni, kapsamı, atıf ve yeniden dağıtım koşulları doğrulanmayı bekliyor. Açık erişim bağlantısı veya GitHub kod lisansı veri lisansı yerine geçmez. Arşivde bulunan LICENSE/README adayları raporlanır; otomatik lisans kabulü yapılmaz.

**İndirme politikası:** Tam arşiv ve örnek arşiv için indirme kodu yoktur. Örneğin yaklaşık 971,6 MB indirmesi de önceden kullanıcı onayı gerektirir; bu notebook yalnızca kullanıcının önceden sağladığı ZIP'i okur.


### Deney planı ve HIT-UAV ile karşılaştırma ilkeleri
1. Resmî örnek arşiv kimliği/sürümü, kullanım koşulları ve varsa yayıncı checksum'ını kaydedin.
2. Önce dosya ağacını, gerçek boyut/mod/kanalları ve ham etiketleri inceleyin. RGB görüntü modu, RGB sensör modalitesini kanıtlamaz; sahte renkli termal de RGB olabilir.
3. Eşleştirmeyi resmî zaman/eşleme bilgisi ve gerçek dosya örnekleriyle doğrulayın. Dosya sırası veya eşit dosya adı tek başına senkronizasyon/hizalama kanıtı değildir.
4. Örnek tek uçuşa aittir. Yakın karelerin ve eşzamanlı modalite eşlerinin train/test'e dağılması sızıntı riski taşır. Mevcut split/flight metadata bulunmadan uçuş ID'si üretmeyin. **Yeni split oluşturulmaz.**
5. Örnekten tam veri dağılımı veya uçuşlar arası genelleme sonucu çıkarılmaz. HIT-UAV ile sensör, perspektif, çözünürlük, sınıf ve kutu temsil farklarını belgeleyin. Eğitim ve bilimsel değerlendirme planı kullanıcı onayını bekler.

Aşağıdaki kod kaynak planını ve karşılaştırma tablosunu arşiv olmadan da üretir. Yeni bağımlılık gerekmez; yalnızca Pillow/Matplotlib kullanılır. Colab'ın ipykernel/jupyter-client paketleri değiştirilmez.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

REPOSITORY = 'https://github.com/hilalaltunayy/therma-search-ai.git'
CODE_REF = '48bef665e2c0cc8fe4bd3bd221a21b7fdc0ddc30'  # Sınanan WiSARD yardımcı kod commit'i.
local_code_root = os.environ.get('THERMA_CODE_ROOT')
if local_code_root:
    PROJECT_ROOT = Path(local_code_root).resolve()
else:
    PROJECT_ROOT = Path('/content/therma-search-ai-wisard')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git','clone','--quiet',REPOSITORY,str(PROJECT_ROOT)],check=True)
    remote = subprocess.check_output(['git','-C',str(PROJECT_ROOT),'remote','get-url','origin'],text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('Mevcut kod klasörünün kaynağı beklenen depo değil.')
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Yerel kod değişikliği var; üzerine yazılmadı. Temiz çalışma zamanı kullanın.')
    subprocess.run(['git','-C',str(PROJECT_ROOT),'fetch','--quiet','--depth','1','origin',CODE_REF],check=True)
    subprocess.run(['git','-C',str(PROJECT_ROOT),'checkout','--quiet','--detach','FETCH_HEAD'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','-r',str(PROJECT_ROOT/'requirements-colab.txt')],check=True)
sys.path.insert(0,str(PROJECT_ROOT))
CODE_REVISION = subprocess.check_output(['git','-C',str(PROJECT_ROOT),'rev-parse','HEAD'],text=True).strip()
from src.data.wisard import SOURCE, PairingSpec, LabelSpec, sample_candidates
from src.data.wisard_reports import run_wisard_exploration, comparison_rows
from IPython.display import display, HTML, Image as DisplayImage
from html import escape

def show_table(rows, columns):
    display(HTML('<table><tr>'+''.join('<th>'+escape(c)+'</th>' for c in columns)+'</tr>'+''.join(
        '<tr>'+''.join('<td>'+escape(str(row.get(c,'')))+'</td>' for c in columns)+'</tr>' for row in rows)+'</table>'))
print('Kod commit:',CODE_REVISION)
print('Lisans:',SOURCE['license'])
show_table(comparison_rows(),['field','HIT_UAV','WiSARD'])

## Aşama 2 — Önceden sağlanan örnek ZIP (isteğe bağlı)
Colab'da `CONNECT_DRIVE=True` Drive bağlantısı açar; yalnızca raporlar `ThermaSearchAI/reports/wisard/` içine yazılır. Mevcut veri dosyaları değiştirilmez.

`SAMPLE_ZIP_PATH` biliniyorsa kendi gerçek yolunuzu girin. Varsayılan `None`: proje veri dizinindeki mevcut, adında WiSARD geçen ZIP'ler **yalnızca listelenir**. Uygun boyutlu tek aday varsa seçilir; birden fazla adayda seçim yapılmaz. Listeleme dağıtım sürümünü veya arşivin gerçekten örnek olduğunu doğrulamaz; dosya kaynağını kullanıcı kontrol etmelidir.

ZIP yoksa kaynak/deney planı ve karşılaştırma raporuyla tamamlanır. Dosya adı veya arşiv içindeki klasör adı tahmin edilmez. 2 GiB'den büyük ZIP reddedilir; 40,54 GB tam arşivi göstermeyin. 8 GiB toplam açılmış boyut, 32 MiB üye ve 25 milyon piksel limiti uygulanır; arşiv çıkarılmaz.

Drive bağlamadan sadece Aşama 1'i çalıştırmak için `CONNECT_DRIVE=False` seçebilirsiniz. Bu durumda raporlar yerel `outputs/wisard/` altında kalır ve Colab kapanınca kaybolabilir; Drive'a kalıcı kayıt yapılmış sayılmaz.


In [ ]:
CONNECT_DRIVE = True
if not local_code_root and CONNECT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path(os.environ.get('THERMA_WISARD_DATA_DIR','/content/drive/MyDrive/ThermaSearchAI/datasets/'))
REPORT_ROOT = Path(os.environ.get('THERMA_WISARD_REPORT_ROOT',
    '/content/drive/MyDrive/ThermaSearchAI/reports/wisard/' if CONNECT_DRIVE else str(PROJECT_ROOT/'outputs/wisard')))
SAMPLE_ZIP_PATH = os.environ.get('THERMA_WISARD_ZIP') or None  # Varsa kendi gerçek mevcut ZIP yolunuzu girebilirsiniz.
candidates = sample_candidates(DATA_DIR)
print('Mevcut adaylar:',candidates)
if SAMPLE_ZIP_PATH is None:
    eligible = [r for r in candidates if r['eligible_sample_size']]
    if len(eligible) == 1:
        SAMPLE_ZIP_PATH = eligible[0]['path']
    elif len(eligible) > 1:
        print('Birden fazla örnek adayı var; SAMPLE_ZIP_PATH seçmeden arşiv analizi yapılmayacak.')
if SAMPLE_ZIP_PATH is not None and not Path(SAMPLE_ZIP_PATH).is_file():
    raise FileNotFoundError('Seçilen dosya mevcut değil; indirme yapılmayacak.')
print('Arşiv:',SAMPLE_ZIP_PATH or 'Yok; yalnızca kaynak planı üretilecek')
print('Rapor kökü:',REPORT_ROOT)

### Yapı ve şema doğrulama kapısı
**İlk çalıştırmada bu hücreyi değiştirmeyin.** Eşleştirme ve kutu gösterimi kapalı kalır; envanter/ham önizlemelerden gerçek yapıyı inceleyin. Daha sonra aşağıdaki kuralları gerçek gözlemler ve kaynak kanıtıyla doldurup yalnızca bu hücreyi ve sonrasını tekrar çalıştırın. Yeni run klasörü oluşur; önceki rapor korunur.

`PairingSpec`: RGB ve termal için ZIP içindeki **tam göreli yola** uygulanan iki regex, aynı kareyi tanımlayan `key` adlı grup ve `evidence` gerektirir. Anahtar birden fazla uçuş varsa uçuşu da ayırmalı; aynı anahtarlı birden fazla dosya varsa ilk dosya seçilmez, sorun raporlanır. Regex'lerde isteğe bağlı `flight`/`split` grupları yalnızca doğrulanmamış yol tokenlarıdır. Bunların splitler arası örtüşmesi risk adayı olarak raporlanır; gerçek uçuş/split kanıtı değildir.

`LabelSpec`: format doğrulanmadan sınıf sayımı veya Person kutusu yoktur. Desteklenen tek adaptör, kullanıcı tarafından açıkça doğrulanmış **5 sütunlu `class cx cy w h`, normalize merkez tabanlı TXT** (`yolo_cxcywh_normalized`). `image_regex` içindeki adlandırılmış gruplardan `label_template` ile gerçek etiket yolu oluşturulur; `class_names` sözlüğü ve `person_class_ids` kullanıcı tarafından doğrulanmalıdır. Burada gerçek WiSARD sınıf ID'si varsayılmamıştır. Aynı etiketin RGB/termal iki görüntü için çift sayılması hata olarak raporlanır.

JSON/XML, polygon, OBB, piksel koordinatı veya başka bir şema gözlenirse `confirmed=True` yapıp bu adaptörü kullanmayın; gerçek örneğe dayalı ayrı adaptör gerekir. Desteklenmeyen biçim sessizce dönüştürülmez. Sınır dışı kutular raporlanır, kırpılmaz. Boş etiket geçerli negatif olarak kabul edilmez.


In [ ]:
PAIRING_SPEC = PairingSpec()  # confirmed=False; regex/kanıt gerçek arşiv görüldükten sonra girilir.
LABEL_SPEC = LabelSpec()      # confirmed=False; sınıf/Person ID ve temsil varsayılmaz.
MAX_PREVIEWS = 3
EXTRA_PREVIEW_MEMBERS = []  # Envanterden gözlenen gerçek metadata/split dokümanı yolları; en fazla 12.

### Envanter ve kalıcı raporu üret
Arşiv varsa tüm aday görüntülerin gerçek çözünürlük/mod/kanal bilgileri okunur. TXT/JSON/XML/CSV dosyaları etiket **adayıdır**; README veya split listesi de olabilir. Önizlemeler en fazla 64 KiB okur, ilk 3000 karakteri saklar; kısmi önizleme tam dosya bütünlüğü doğrulaması değildir. Lisans dosyası bulunması koşulları otomatik kabul etmez.

Her çalıştırma `run_wisard_.../` klasörüne kaynak/plan JSON'u, HIT-UAV karşılaştırma CSV'si ve özet yazar. ZIP varsa envanter/kanal-boyut CSV'si, ham etiket/lisans önizlemeleri, yol ipuçları, eşleştirme ve anotasyon raporları da yazılır. Onaylı eşleştirmede en fazla 3 yan yana çift, onaylı Person şemasında en fazla 3 kutulu örnek kaydedilir; tüm görüntüler tekrar kaydedilmez.


In [ ]:
WISARD_RUN_DIR, WISARD_SUMMARY = run_wisard_exploration(
    REPORT_ROOT, archive_path=SAMPLE_ZIP_PATH,
    pairing_spec=PAIRING_SPEC, label_spec=LABEL_SPEC,
    code_revision=CODE_REVISION, max_previews=MAX_PREVIEWS, preview_members=EXTRA_PREVIEW_MEMBERS,
)
print('Durum:',WISARD_SUMMARY['status'])
print('Raporlar:',WISARD_RUN_DIR)
print('Yerel görüntü sayısı:',WISARD_SUMMARY['local_image_count'])
if WISARD_SUMMARY['archive_inspected']:
    print('Gerçek uzantılar:',WISARD_SUMMARY['extensions'])
    print('Gerçek dizinler (ilk 30):',list(WISARD_SUMMARY['directories'].items())[:30])
    print('Eşleştirme:',WISARD_SUMMARY['pairing_status'],'| kutu şeması:',WISARD_SUMMARY['label_status'])
    print('Uçuş/split token örtüşmesi adayları:',WISARD_SUMMARY['flight_split_overlap_candidates'][:10])
    print('Görüntü hataları (ilk 5):',WISARD_SUMMARY['image_errors'][:5])
else:
    print('Arşiv yok: çözünürlük/kanal/şema/eşleştirme/split/flight yerel olarak doğrulanmadı.')

### Gerçek dosya yapısı ve ham etiket/lisans önizlemeleri
Tam envanter Drive CSV'sindedir; aşağıdaki tablolar sınırlı önizlemedir. Kanal sayısı modalite, klasör tokenı uçuş ID'si değildir. Split üyelik listeleri veya uçuş metadata'sı varsa önizlemesini ve tam kaydını ayrıca okuyun; desteklenmeyen şemadan üyelik üretmeyin.


Özel metadata/split belgesi ilk otomatik önizlemelerde yoksa gerçek üye yolunu envanterden seçip EXTRA_PREVIEW_MEMBERS listesine ekleyin; kural hücresinden itibaren yeniden çalıştırın. Yeni raporda sınırlı metin önizlemesi saklanır; üyelikler otomatik yorumlanmaz.

In [ ]:
import csv
for filename, columns in [('archive_inventory.csv',['member','bytes','extension','parent']),
                          ('image_metadata.csv',['member','width','height','mode','channels','status'])]:
    path = WISARD_RUN_DIR/filename
    if path.exists():
        with path.open(encoding='utf-8-sig') as stream:
            show_table(list(csv.DictReader(stream))[:12],columns)
for filename in ('label_previews.json','license_candidates.json','extra_metadata_previews.json'):
    path = WISARD_RUN_DIR/filename
    if path.exists():
        print(filename,json.dumps(json.loads(path.read_text(encoding='utf-8')),ensure_ascii=False,indent=2)[:9000])
print('Karşılaştırma:')
with (WISARD_RUN_DIR/'hit_uav_comparison.csv').open(encoding='utf-8-sig') as stream:
    show_table(list(csv.DictReader(stream)),['field','HIT_UAV','WiSARD'])

### Yalnızca doğrulanmış kurallarla görselleştirme
Kural kapalıysa sahte RGB/termal çift veya Person kutusu üretilmez. Yan yana çizim geometrik hizalama değildir; farklı çözünürlükler yeniden ölçeklenip tek kutu uzayına taşınmaz. Sınıf grafiği yalnızca onaylı eşlemede ayrıştırılan etiket satırlarını sayar; eksik dosyalar/bozuk satırlar raporda kalır.


In [ ]:
pngs = sorted(WISARD_RUN_DIR.glob('*.png'))
for path in pngs:
    display(DisplayImage(filename=str(path)))
if not pngs:
    print('Görsel üretilmedi: arşiv ve kullanıcı doğrulamalı eşleştirme/etiket kuralları bekleniyor.')
for filename in ('pairing_report.json','annotation_report.json'):
    path = WISARD_RUN_DIR/filename
    if path.exists():
        report = json.loads(path.read_text(encoding='utf-8'))
        print(filename,'durum:',report['status'],'ilk 5 sorun:',report['issues'][:5])
print(*WISARD_SUMMARY['limitations'],sep='\n- ')

## Sonraki adım ve açık araştırma konuları
Önce resmî veri lisansını ve örnek dağıtım kimliğini doğrulayın. İndirme gerekiyorsa **önceden kullanıcı onayı alın**; bu notebook bunu otomatik yapmaz. Mevcut örnekte gerçek dosya ağacı ve ham etiketleri inceleyip yalnızca doğruladığınız kuralları açın.

Tek uçuş örneğini rastgele train/test ayırmak yakın kare/aynı sahne sızıntısını önlemez. Mevcut uçuş/split metadata'sını resmî kaynakla doğrulayın; görüntü benzerliği veya yol tokenı gerçek uçuş kimliği yerine geçmez. Yakın kare/sızıntı taraması ve bağımsız uçuş değerlendirmesi bu keşifte yapılmış sayılmaz. Yeni split veya model eğitimi başlatılmaz. HIT-UAV ile karşılaştırma tablolarında kaynak beyanını ve örnek ölçümünü ayrı tutun.
